# N086 · 最小生成树与割性质

**目标：** 区分连接全部顶点的最小总成本和最短路。

**先修：** N083, N084。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** Kruskal；Prim；割性质；环性质；DSU；图不连通。

## 从问题到算法

最小生成树连接所有顶点且无环，不等同于从某源的最短路径树。割性质说明：跨越某个割的最轻边，可以加入某棵MST。Kruskal按边权连接不同分量；Prim维护已经连接的一侧与外侧之间的最轻边。断开图不存在生成树，应明确报错而不是返回一片森林成本。

## 最小实现

**本章接口：** `kruskal(n, edges)`、`prim(n, adj)`、`min_cost_connect_points(points)`

In [1]:
class DSU:
    def __init__(self,n): self.parent=list(range(n)); self.size=[1]*n; self.components=n
    def find(self,x):
        while self.parent[x]!=x:
            self.parent[x]=self.parent[self.parent[x]]; x=self.parent[x]
        return x
    def union(self,a,b):
        a=self.find(a); b=self.find(b)
        if a==b: return False
        if self.size[a]<self.size[b]: a,b=b,a
        self.parent[b]=a; self.size[a]+=self.size[b]; self.components-=1
        return True

import heapq

def kruskal(n,edges):
    dsu=DSU(n); chosen=[]; total=0
    for u,v,w in sorted(edges,key=lambda e:e[2]):
        if dsu.union(u,v): chosen.append((u,v,w)); total+=w
    if n and len(chosen)!=n-1: raise ValueError('graph is disconnected')
    return total,chosen

def prim(n,adj):
    if n==0: return 0,[]
    seen=set(); heap=[(0,0,-1)]; total=0; chosen=[]
    while heap and len(seen)<n:
        weight,u,parent=heapq.heappop(heap)
        if u in seen: continue
        seen.add(u); total+=weight
        if parent!=-1: chosen.append((parent,u,weight))
        for v,w in adj[u]:
            if v not in seen: heapq.heappush(heap,(w,v,u))
    if len(seen)!=n: raise ValueError('graph is disconnected')
    return total,chosen

def min_cost_connect_points(points):
    n=len(points)
    if not n: return 0
    best=[float('inf')]*n; best[0]=0; used=[False]*n; total=0
    for _ in range(n):
        u=min((i for i in range(n) if not used[i]),key=lambda i:best[i])
        total+=best[u]; used[u]=True
        for v in range(n):
            if not used[v]: best[v]=min(best[v],abs(points[u][0]-points[v][0])+abs(points[u][1]-points[v][1]))
    return total

## 正确性、前提与复杂度

若MST不含选定割的最轻边，加入它会形成环，环上另一条跨割边不轻于它；交换不会增大成本。反复交换说明贪心选择可嵌入某棵MST。

**代价：** Kruskal O(E log E)，堆Prim O(E log E)；完全点图的本章稠密Prim O(n²)时间/O(n)空间，避免显式保存全部O(n²)边。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

edges=[(0,1,1),(1,2,2),(0,2,3)]; total,chosen=kruskal(3,edges)
show_table(['选择次序','边','成本'],[(i,(u,v),w) for i,(u,v,w) in enumerate(chosen)])
show_table(['总成本'],[(total,)])

选择次序,边,成本
0,"(0, 1)",1
1,"(1, 2)",2


总成本
3


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert kruskal(1,[])[0]==0
assert kruskal(3,[(0,1,1),(1,2,2),(0,2,3)])[0]==3
assert min_cost_connect_points([[0,0],[2,2],[3,10],[5,2],[7,0]])==20
from itertools import combinations
from random import Random
rng=Random(86)
for n in range(2,6):
    for _ in range(12):
        edges=[(u,v,rng.randrange(-2,6)) for u,v in combinations(range(n),2)]
        adj=[[] for _ in range(n)]
        for u,v,w in edges: adj[u].append((v,w)); adj[v].append((u,w))
        ref=float('inf')
        for subset in combinations(edges,n-1):
            reach={0}
            for _ in range(n):
                for u,v,_ in subset:
                    if u in reach or v in reach: reach.update([u,v])
            if len(reach)==n: ref=min(ref,sum(w for _,_,w in subset))
        assert kruskal(n,edges)[0]==prim(n,adj)[0]==ref
print('N086: 所有本章断言通过')

N086: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 构造MST路径不是最短路径的反例。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 处理不连通图并明确返回契约。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1584. Min Cost to Connect All Points（canonical）
- 1489. Find Critical and Pseudo-Critical Edges in Minimum Spanning Tree（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。